# PyMC-NMR Tutorial 2: Comparing Modes of Operation

Welcome to the second tutorial! In this guide, we will compare the three major structure-search modes of **PyMC-NMR**:
1. **Monte Carlo** (`monte`)
2. **Basin Hopping** (`basinhop`)
3. **AIRSS-Style Random Search** (`airss`)

We will explore how to configure each, run them pythonically, and compare their performance characteristics.

## 1. Monte Carlo Mode (`monte`)

In standard Monte Carlo, single atoms are translated or swapped, and the Metropolis acceptance rule is applied based *immediately* on the raw potential energy change.

**Pros:** Per-step computation is incredibly fast (no geometry optimization).
**Cons:** High probability of rejection if swaps cause high steric clashes/high energy barriers.

In [ ]:
import yaml
import sys
from pymc_nmr.monte import run_simulation

mc_config = {
    "job": {
        "structure_method": "monte",
        "temperature": 1300.0
    },
    "potential": {
        "arch": "mace_mp",
        "model": "MACE-matpes-r2scan-omat-ft.model",
        "device": "cpu"
    },
    "monte": {
        "steps": 5
    },
    "moves": {
        "swaps": [
            {
                "pairs": [["Al", "Si"]],
                "frequency": 100
            }
        ]
    }
}

with open("config_mc.yaml", "w") as f:
    yaml.safe_dump(mc_config, f)

# Run Monte Carlo!
run_simulation(config_path="config_mc.yaml", log_path="mc_monte.log")


## 2. Basin Hopping Mode (`basinhop`)

In Basin Hopping, every substitution swap is immediately followed by a complete **geometry minimisation** (structural relaxation). The energy is computed *after* relaxation, and Metropolis sampling is performed on these minimized state energies.

**Pros:** Exceptional search capability. Easily navigates large barriers/clashes by relaxing atoms first.
**Cons:** Significantly higher computational cost per step due to relaxation cycles.

In [ ]:
bh_config = {
    "job": {
        "structure_method": "basinhop",
        "temperature": 1200.0
    },
    "potential": {
        "arch": "mace_mp",
        "model": "MACE-matpes-r2scan-omat-ft.model",
        "device": "cpu"
    },
    "monte": {
        "steps": 5
    },
    "moves": {
        "swaps": [
            {
                "pairs": [["Al", "Si"]],
                "counterion": "H",
                "max_distance": 4.0,
                "frequency": 100
            }
        ]
    },
    "relax": {
        "method": "bfgs",
        "steps": 100,
        "tol": 0.05
    }
}

with open("config_bh.yaml", "w") as f:
    yaml.safe_dump(bh_config, f)

# Run Basin Hopping!
run_simulation(config_path="config_bh.yaml", log_path="mc_bh.log")


## 3. AIRSS-Style Mode (`airss`)

Ab Initio Random Structure Search completely randomizes all Al/Si framework symbols in the lattice at each cycle, followed by relaxation.

**Pros:** Unbiased generation of extremely different structures.
**Cons:** Tends to be less efficient at finding local defect minimums compared to systematic swaps.

In [ ]:
airss_config = {
    "job": {
        "structure_method": "airss",
        "temperature": 1200.0
    },
    "potential": {
        "arch": "mace_mp",
        "model": "MACE-matpes-r2scan-omat-ft.model",
        "device": "cpu"
    },
    "monte": {
        "steps": 5
    },
    "moves": {
        "swaps": [
            {
                "pairs": [["Al", "Si"]],
                "frequency": 100
            }
        ]
    },
    "relax": {
        "method": "bfgs",
        "steps": 100,
        "tol": 0.05
    }
}

with open("config_airss.yaml", "w") as f:
    yaml.safe_dump(airss_config, f)

# Run AIRSS!
run_simulation(config_path="config_airss.yaml", log_path="mc_airss.log")


## Summary of Best Practices

| Method | Step Speed | Acceptance Rate | Primary Use Case |
| :--- | :--- | :--- | :--- |
| **Monte Carlo** | Very Fast | Low | High-temperature structural runs, soft structures, or low energy barriers. |
| **Basin Hopping** | Slow | Very High | Rigid materials, substitutions with major charge changes (e.g. Al substitution in zeolites). |
| **AIRSS** | Very Slow | N/A (Random) | Creating entirely new un-biased starting configurations. |